# Technique: data poisoning with image data

Based on the following paper:

**N. Narodytska and S. Kasiviswanathan, "Simple Black-Box Adversarial Attacks on Deep Neural Networks," 2017 IEEE Conference on Computer Vision and Pattern Recognition Workshops (CVPRW), Honolulu, HI, USA, 2017, pp. 1310-1318, doi: 10.1109/CVPRW.2017.172. keywords: {Knowledge engineering;Training;Neural networks;Network architecture;Cats;Robustness;Computer vision}.**

This part of the project aims to try and (semi-)replicate results presented in the above paper using a CNN, and compare classifications between the unaffected CNN and the attacked CNN. Here we'll operate using their definition:

**Definition.**
A neural network NN **$k$-
misclassifies** an image $I$ with true label $c(I)$ iff the output
$NN(I)$ of the network satisfies $c(I) \notin   \pi(NN(I),k)$, where $\pi$ is a function which returns a set of $k$-labels.

Since the neural network is attacked at test-time, this is an example of a block-box-style attack.

In [7]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as dsets

'''
CNN.ipynb
----------

Implements a CNN (using the implementation from class with some tweaks.) Performs
rudimentary data analysis using the MNIST dataset, and then attempts to perturb
the dataset so our CNN will misclassify the image. The perturbations should be as minor
and undetectable as possible.

'''

train_dataset = dsets.MNIST(root='./data',
                            train=True,
                            transform=transforms.ToTensor(),
                            download=True)

test_dataset = dsets.MNIST(root='./data',
                           train=False,
                           transform=transforms.ToTensor())
# make dataset iterable
batch_size = 100
n_iters = 3000
num_epochs = n_iters / (len(train_dataset) / batch_size)
num_epochs = int(num_epochs)

train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                           batch_size=batch_size,
                                           shuffle=True)

test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                          batch_size=batch_size,
                                          shuffle=False)


def misclassification_attack(distance, Xtrain, perturbation_parameter):
  '''
  Takes in perturbation parameter epsilon and returns an
  altered image coordinate
  '''


  return


# this will be the NN that we'll be attacking
class victim_CNN(nn.Module):
  def __init__(self):
    super(victim_CNN, self).__init__()

    # Convolution 1
    self.cnn1 = nn.Conv2d(in_channels = 1, out_channels = 16, kernel_size=5, stride=1, padding=2)
    self.relu1 = nn.ReLU()

    # Max pool 1
    self.maxpool1 = nn.MaxPool2d(kernel_size = 2)

    # Convolution 2
    self.cnn2 = nn.Conv2d(in_channels = 16, out_channels = 32, kernel_size = 5, stride = 1, padding = 2)
    self.relu2 = nn.ReLU()

    # Max pool 2
    self.maxpool2 = nn.MaxPool2d(kernel_size = 2)

    self.fc1 = nn.Linear(32 * 7 * 7, 10)

  def forward(self, x):
    # input: x, size (num_img, 28, 28)

    # Convolution 1
    # O = (28 - 5 + 2*2)/ 1 + 1 = 28
    # output: size (num_img, 16, 28, 28)
    out = self.cnn1(x)
    out = self.relu1(out)

    # Max pool 1
    # O = 28 / 2 = 14
    # output: size (num_img, 16, 14, 14)
    out = self.maxpool1(out)

    # Convolution 2
    # O = (14 - 5 + 2*2)/ 1 + 1 = 14
    # output: size (num_img, 32, 14, 14)
    out = self.cnn2(out)
    out = self.relu2(out)

    # Max pool 2
    # O = 14 / 2 = 7
    # output: size (num_img, 32, 7, 7)
    out = self.maxpool2(out)

    # Resize
    # Original size: (num_img, 32, 7, 7)
    # out.size(0): num_img
    # New out size: (num_img, 32*7*7)
    out = out.view(out.size(0), -1)

    # Linear function (readout)
    # output: size (num_img, 10)
    out = self.fc1(out)

    return out

model = victim_CNN()
criterion = nn.CrossEntropyLoss()
learning_rate = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

iter = 0
for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        # Load images
        images = images.requires_grad_()

        # Clear gradients w.r.t. parameters
        optimizer.zero_grad()

        # Forward pass to get output/logits
        outputs = model(images)

        # Calculate Loss: softmax --> cross entropy loss
        loss = criterion(outputs, labels)

        # Getting gradients w.r.t. parameters
        loss.backward()

        # Updating parameters
        optimizer.step()

        iter += 1

        if iter % 500 == 0:
            # Calculate Accuracy
            correct = 0
            total = 0
            # Iterate through test dataset
            for images, labels in test_loader:
                # Load images
                images = images.requires_grad_()

                # Forward pass only to get logits/output
                outputs = model(images)

                # Get predictions from the maximum value
                _, predicted = torch.max(outputs.data, 1)

                # Total number of labels
                total += labels.size(0)

                # Total correct predictions
                correct += (predicted == labels).sum()

            accuracy = 100 * correct / total

            # Print Loss
            print('Iteration: {}. Loss: {}. Accuracy: {}'.format(iter, loss.item(), accuracy))




Iteration: 500. Loss: 0.48948830366134644. Accuracy: 88.43000030517578
Iteration: 1000. Loss: 0.24081426858901978. Accuracy: 93.41999816894531
Iteration: 1500. Loss: 0.14413906633853912. Accuracy: 94.97000122070312
Iteration: 2000. Loss: 0.1559636890888214. Accuracy: 95.44999694824219
Iteration: 2500. Loss: 0.1746739149093628. Accuracy: 96.62999725341797
Iteration: 3000. Loss: 0.175285205245018. Accuracy: 96.87000274658203
